In [0]:
%run ./PROJECT_1_word_count_batch_processing

In [0]:
project_dir="dbfs:/FileStore/project1/"

dbutils.fs provides utilities for working with FileSystems. Most methods in
this package can take either a DBFS path (e.g., "/foo" or "dbfs:/foo"), or
another FileSystem URI.

For more info about a method, use dbutils.fs.help("methodName") .

In notebooks, you can also use the %fs shorthand to access DBFS. The %fs shorthand maps
straightforwardly onto dbutils calls. For example, "%fs head --maxBytes=10000 /file/path"
translates into "dbutils.fs.head("/file/path", maxBytes = 10000)".
 mount mount(source: String, mountPoint: String, encryptionType: String = "", owner: String = null, extraConfigs: Map = Map.empty[String, String]): boolean -> Mounts the given source directory into DBFS at the given mount point mounts: Seq -> Displays information about what is mounted within DBFS refreshMounts: boolean -> Forces all machines in this cluster to refresh their mount cache, ensuring they receive the most recent information unmount(mountPoint: String): boolean -> Deletes a DBFS mount point updateMount(source: String, mountPoint: String, encryptionType: String = "", owner: String = null, extraConfigs: Map = Map.empty[String, String]): boolean -> Similar to mount(), but updates an existing mount point (if present) instead of creating a new one fsutils cp(from: String, to: String, recurse: boolean = false): boolean -> Copies a file or directory, possibly across FileSystems head(file: String, maxBytes: int = 65536): String -> Returns up to the first 'maxBytes' bytes of the given file as a String encoded in UTF-8 ls(dir: String): Seq -> Lists the contents of a directory mkdirs(dir: String): boolean -> Creates the given directory if it does not exist, also creating any necessary parent directories mv(from: String, to: String, recurse: boolean = false): boolean -> Moves a file or directory, possibly across FileSystems put(file: String, contents: String, overwrite: boolean = false): boolean -> Writes the given String out to a file, encoded in UTF-8 rm(dir: String, recurse: boolean = false): boolean -> Removes a file or directory

In [0]:
class test_word_count_project1(WordCountProject):
    def __init__(self,project_dir):
        self.project_file_dir=project_dir
        self.dataset_dir="dataset/"
        self.landing_zone="landing_zone/"
        self.load_table_name="wordCountTable"   
        self.check_point_dir="check_point"
        self.resul_df=None


    def ingest_file_with_file_name(self, filename:str):
        # copy the text data files from dataset dir to the landing zone
        res=dbutils.fs.cp(self.project_file_dir+self.dataset_dir+filename, self.project_file_dir+self.landing_zone)
        res_str="Text file is copied from the dataset dir to the landing zone" if res else "Copying process failed !"
        print("INGESTION RES:",res_str)
    
    def asser_result(self, expected):
        actual= spark.sql(f"select sum(count) from {self.load_table_name} where substr(word, 1,1)=='s'").collect()[0][0]
        assert expected==actual, f"test failed ! actual value: {actual}  and expected: {expected}"

    def run_test(self, file_name:str, actual_word_count_start_with_s:int):

        #run cleanup
        self.cleanup_n_setup()

        # ingest the data from dataset to landing zone
        self.ingest_file_with_file_name(file_name)

        #Start ETL process
        # Extract data 
        extracted_data_df=self.extract_data_from_landing_zone()

        # transform data
        transformed_df=self.transform_data(extracted_data_df)
        self.resul_df=transformed_df

        # Load data to sink
        self.load_data(transformed_df)
        

        self.asser_result(actual_word_count_start_with_s)






        

In [0]:
wc_proj_test=test_word_count_project1(project_dir=project_dir)
file_value_list=[
    ("text_data_1.txt", 25),
    ("text_data_2.txt", 7),
    ("text_data_3.txt", 5)
]
for file_name, value in file_value_list:
    print("-"*10, f"STARTING TEST CASE: PARAMS (file_name: {file_name}, count_of_words_starting_with_s:{value})", "-"*10)
    wc_proj_test.run_test(file_name, value)
    print("TEST RES: PASSED")



---------- STARTING TEST CASE: PARAMS (file_name: text_data_1.txt, count_of_words_starting_with_s:25) ----------
CLEANUP & SETUP RES: data cleanup and setup successfull !
INGESTION RES: Text file is copied from the dataset dir to the landing zone
EXTRACTION RES: data extraction successfull
TRANSFOMATION RES: data transformation successfull !
LOAD RES: data load to delta-table successfully ! 
TABLE NAME: wordCountTable
TEST RES: PASSED
---------- STARTING TEST CASE: PARAMS (file_name: text_data_2.txt, count_of_words_starting_with_s:7) ----------
CLEANUP & SETUP RES: data cleanup and setup successfull !
INGESTION RES: Text file is copied from the dataset dir to the landing zone
EXTRACTION RES: data extraction successfull
TRANSFOMATION RES: data transformation successfull !
LOAD RES: data load to delta-table successfully ! 
TABLE NAME: wordCountTable
TEST RES: PASSED
---------- STARTING TEST CASE: PARAMS (file_name: text_data_3.txt, count_of_words_starting_with_s:5) ----------
CLEANUP & S